- *Can we identify parallel roles purely from network structure, without narrative context?* SØREN
    * Notes:
        - Node2vec algorithm for finding nodes which are similar in one universe.
        - Struc2vec algorithm for finding nodes across networks which occupy similar structural positions.
        - Maybe: Compute a vector of structural properties (In-degree, Out-degree, Closeness, Eigenvector / pageRank, Clustering coefficient, triadic position, etc.) for each characters, and use clustering algorithms (k-means, hierarchical clustering, Gaussian mixture models)

In [ ]:
# Create the edgelist for the stuc2vec algorithm

import numpy as np
import networkx as nx 
import pickle
from util import *

with open("hp_characters.pkl", "rb") as f:   # 'rb' = read binary
    HP_data = pickle.load(f)
    
HP_network = create_network(HP_data, field_origin = 'house', field_species = 'species')

with open("lotr_characters.pkl", "rb") as f:   # 'rb' = read binary
    LOTR_data = pickle.load(f)

LOTR_network = create_network(LOTR_data, field_origin = 'culture', field_species = 'race')

G = nx.union(HP_network, LOTR_network)
mapping = {old_label: i + 1 for i, old_label in enumerate(G.nodes())}
G = nx.relabel_nodes(G, mapping)
#nx.write_edgelist(G, "C:\\Users\\soere\\OneDrive\\Skrivebord\\struc2vec-master\\struc2vec-master\\graph\\graph.edgelist", data=False)

#with open("mapping.pkl", "wb") as f:
#    pickle.dump(mapping, f)


In [87]:
with open("mapping.pkl", "rb") as f:
    mapping = pickle.load(f)
# Load the generated embedding
from gensim.models import KeyedVectors
embeddings = KeyedVectors.load_word2vec_format(f"embedding.emb", binary=False)

In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

# -----------------------------------------------------------
# Step 1: Function to select best number of GMM clusters via BIC
# -----------------------------------------------------------

def select_best_gmm_k(X, k_min=2, k_max=15):
    bic_scores = []
    gmms = []
    
    for k in range(k_min, k_max + 1):
        gmm = GaussianMixture(
            n_components=k,
            covariance_type='full',
            random_state=42
        )
        gmm.fit(X)
        bic = gmm.bic(X)
        bic_scores.append(bic)
        gmms.append(gmm)

    best_idx = np.argmin(bic_scores)
    best_k = k_min + best_idx
    best_gmm = gmms[best_idx]
    
    print(f"Best number of clusters (BIC): {best_k}")
    return best_k, best_gmm, bic_scores

In [88]:
embeddings.index_to_key

['884',
 '885',
 '86',
 '88',
 '87',
 '160',
 '79',
 '110',
 '101',
 '229',
 '120',
 '119',
 '98',
 '80',
 '326',
 '327',
 '299',
 '298',
 '27',
 '26',
 '535',
 '1207',
 '97',
 '305',
 '122',
 '121',
 '137',
 '136',
 '138',
 '85',
 '330',
 '171',
 '176',
 '1206',
 '790',
 '533',
 '105',
 '251',
 '300',
 '84',
 '16',
 '264',
 '608',
 '490',
 '82',
 '158',
 '607',
 '237',
 '858',
 '704',
 '318',
 '28',
 '109',
 '267',
 '100',
 '32',
 '30',
 '63',
 '55',
 '93',
 '83',
 '31',
 '331',
 '69',
 '249',
 '168',
 '144',
 '483',
 '436',
 '142',
 '823',
 '166',
 '208',
 '123',
 '908',
 '13',
 '524',
 '193',
 '104',
 '185',
 '539',
 '218',
 '506',
 '283',
 '11',
 '10',
 '17',
 '873',
 '18',
 '938',
 '301',
 '910',
 '1099',
 '228',
 '78',
 '266',
 '344',
 '567',
 '155',
 '172',
 '156',
 '1256',
 '852',
 '25',
 '961',
 '414',
 '415',
 '1188',
 '227',
 '34',
 '616',
 '19',
 '23',
 '555',
 '391',
 '6',
 '308',
 '1253',
 '77',
 '511',
 '662',
 '996',
 '111',
 '47',
 '1006',
 '152',
 '196',
 '645',
 '806

In [85]:
[(node, deg) for node, deg in G.in_degree()]

[(1, 0),
 (2, 40),
 (3, 86),
 (4, 201),
 (5, 83),
 (6, 1),
 (7, 25),
 (8, 8),
 (9, 101),
 (10, 1),
 (11, 1),
 (12, 12),
 (13, 4),
 (14, 6),
 (15, 7),
 (16, 4),
 (17, 5),
 (18, 1),
 (19, 4),
 (20, 10),
 (21, 9),
 (22, 5),
 (23, 4),
 (24, 11),
 (25, 3),
 (26, 2),
 (27, 2),
 (28, 1),
 (29, 20),
 (30, 0),
 (31, 0),
 (32, 0),
 (33, 3),
 (34, 3),
 (35, 7),
 (36, 35),
 (37, 18),
 (38, 20),
 (39, 26),
 (40, 11),
 (41, 12),
 (42, 87),
 (43, 34),
 (44, 21),
 (45, 22),
 (46, 33),
 (47, 1),
 (48, 17),
 (49, 3),
 (50, 9),
 (51, 8),
 (52, 9),
 (53, 0),
 (54, 3),
 (55, 5),
 (56, 5),
 (57, 8),
 (58, 3),
 (59, 13),
 (60, 5),
 (61, 14),
 (62, 6),
 (63, 1),
 (64, 19),
 (65, 13),
 (66, 7),
 (67, 11),
 (68, 11),
 (69, 1),
 (70, 4),
 (71, 8),
 (72, 8),
 (73, 7),
 (74, 13),
 (75, 4),
 (76, 12),
 (77, 3),
 (78, 0),
 (79, 5),
 (80, 5),
 (81, 6),
 (82, 4),
 (83, 1),
 (84, 2),
 (85, 1),
 (86, 2),
 (87, 2),
 (88, 2),
 (89, 9),
 (90, 3),
 (91, 0),
 (92, 1),
 (93, 0),
 (94, 4),
 (95, 0),
 (96, 4),
 (97, 0),
 (98, 0

In [90]:
nx.closeness_centrality(G)

{1: 0.0,
 2: 0.15375449220001125,
 3: 0.16524876006350722,
 4: 0.19275902929265284,
 5: 0.16381734635747108,
 6: 0.10278153554674666,
 7: 0.14522715261553962,
 8: 0.11423236433920297,
 9: 0.16605485157601213,
 10: 0.000723589001447178,
 11: 0.000723589001447178,
 12: 0.0846797128683644,
 13: 0.09255368290669518,
 14: 0.12524372543444623,
 15: 0.1253359520363862,
 16: 0.09255368290669518,
 17: 0.12515163445986208,
 18: 0.06824628021868982,
 19: 0.08359834128949531,
 20: 0.11778977360928196,
 21: 0.10945737804849674,
 22: 0.09551415424546152,
 23: 0.08359834128949531,
 24: 0.12845752669087732,
 25: 0.002170767004341534,
 26: 0.0016280752532561505,
 27: 0.0016280752532561505,
 28: 0.0013024602026049205,
 29: 0.13605613338562145,
 30: 0.0,
 31: 0.0,
 32: 0.0,
 33: 0.06488990578170509,
 34: 0.06488990578170509,
 35: 0.0670103239627608,
 36: 0.15169895086043889,
 37: 0.14510334430128938,
 38: 0.14473318270868404,
 39: 0.14375525579849024,
 40: 0.11555072835397992,
 41: 0.1218369526595651,
 4

In [ ]:
def get_ordering(ordering):
    return np.array([snd for _, snd in sorted([(node, deg) for node, deg in ordering], key = lambda x: x[0] )])

def construct_X(G, embedding):
    in_deg = get_ordering(G.in_degree())
    out_deg = get_ordering(G.out_degree())
    closesness = get_ordering(nx.closeness_centrality(G)) 
    pagerank = get_ordering(nx.pagerank(G))  
    clustering = get_ordering(nx.clustering(G))  

    word_vectors = []

    for key in embedding.index_to_key:
        word_vectors.append((int(key), embedding[key]))

    word_vectors = sorted(word_vectors, key = lambda x: x[0])

    word_vectors = np.array([snd for _, snd in word_vectors])

    X = np.c_[np.array([in_deg, out_deg, closesness, pagerank, clustering]).T, word_vectors]
    return X

import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def plot_pca_first_two(Z, labels):
    """
    Plots the first two PCA directions of Z, colored by labels.
    
    Parameters
    ----------
    Z : np.ndarray
        Normalized data matrix of shape (n_samples, n_features)
    labels : array-like
        Vector of group labels (length n_samples)
    """
    # PCA → 2D projection
    pca = PCA(n_components=2)
    Z_pca = pca.fit_transform(Z)

    # Convert labels to a numpy array
    labels = np.array(labels)

    # Unique groups
    unique_labels = np.unique(labels)

    plt.figure(figsize=(7, 7))

    # Plot each group separately to color them differently
    for lab in unique_labels:
        idx = labels == lab
        plt.scatter(
            Z_pca[idx, 0],
            Z_pca[idx, 1],
            label=str(lab)
        )

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("PCA (first two components)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
X = construct_X(G, embeddings)
Z = (X - np.mean(X, axis = 0)) / np.std(X, axis = 0)

In [ ]:
best_k, best_gmm, bic_scores = select_best_gmm_k(Z)

result = best_gmm.fit_predict(Z)
reversed_mapping = {value : item for item, value in mapping.items()}
groups = {}
for x in set(result):
    idx = np.where(result == x)[0] + 1
    lst = [reversed_mapping[i] for i in idx]
    groups[x] = {'Names' : lst, 'index' : idx}

plot_pca_first_two(Z, result)

In [ ]:
groups

In [ ]:
Z_main = Z[groups[0]['index'],:]
_, best_gmm_main, _ = select_best_gmm_k(Z_main)
result_main = best_gmm_main.predict(Z_main)
reversed_mapping = {value : item for item, value in mapping.items()}
groups_main = {}
for x in set(result_main):
    idx = np.where(result_main == x)[0] + 1
    lst = [reversed_mapping[i] for i in idx]
    groups_main[x] = {'Names' : lst, 'index' : idx}
plot_pca_first_two(Z_main, result_main)



In [ ]:
groups_main

In [83]:
Z

array([[ 0.20407644,  0.5237192 , -1.3262517 , ..., -0.00482947,
         0.19294976,  0.60493207],
       [-0.7520925 , -0.833226  ,  0.21081637, ..., -1.3279434 ,
         0.7692041 ,  0.33155534],
       [ 1.1434451 ,  0.01769423,  1.8179573 , ...,  0.6551434 ,
        -0.28644866,  2.0987349 ],
       ...,
       [-1.7885121 ,  1.2073611 , -0.7298429 , ...,  0.5798037 ,
        -0.10518206,  0.527645  ],
       [-0.52281237,  1.3343717 , -1.0856844 , ...,  0.35454226,
        -1.0269344 ,  0.88936126],
       [ 0.9409844 , -0.6885534 ,  0.18117584, ..., -1.7303562 ,
        -1.3245136 , -1.1309428 ]], dtype=float32)